# 08 — Análise estatística e consolidação experimental

Este notebook consolida os resultados produzidos pelos notebooks anteriores e transforma as métricas por execução em tabelas comparáveis por cenário.

Até aqui, o fluxo experimental foi dividido em etapas bem separadas:

```text
04 — treinamento dos adaptadores
05 — inferência nos arquivos comuns de avaliação
06 — métricas diretamente computáveis
07 — métricas pareadas e injected-only
```

O objetivo deste notebook é juntar as métricas do `06` e do `07`, organizar os resultados por cenário e seed, calcular médias e dispersões, estimar intervalos de confiança descritivos e produzir tabelas finais de comparação.

Este notebook **não treina modelos** e **não roda inferência**. Ele apenas lê artefatos já produzidos, consolida tabelas e gera novos arquivos de análise.


## 1. Papel deste notebook no experimento

O experimento possui múltiplos cenários e múltiplas seeds. Isso significa que olhar apenas uma linha isolada de resultado pode ser enganoso. Um cenário pode parecer melhor em uma seed, mas pior em outra; uma defesa pode melhorar robustez e, ao mesmo tempo, reduzir utilidade limpa; outra pode preservar utilidade, mas resistir menos aos ataques adaptativos.

Este notebook existe para organizar esse tipo de comparação.

Ele responde principalmente às seguintes perguntas:

```text
- Qual é a média de cada métrica por cenário?
- Qual é a variação entre seeds?
- Quais cenários melhoram em relação ao C0?
- Quais cenários melhoram em relação ao C1 format-only?
- Qual defesa preserva melhor utilidade limpa?
- Qual defesa apresenta maior robustez sob ataque?
- Qual defesa reduz mais o sucesso do ataque?
- As diferenças parecem consistentes entre as seeds?
```

As análises aqui são descritivas. Como o número de seeds é pequeno (`42`, `123` e `2026`), os intervalos de confiança devem ser interpretados com cautela. Eles ajudam a enxergar estabilidade, mas não devem ser tratados como prova estatística forte.


## 2. Imports e diretórios principais

Nesta etapa, são carregadas as bibliotecas necessárias e definidos os diretórios esperados do projeto.

O notebook assume a mesma raiz dos notebooks anteriores:

```text
/workspace/pi-defense-exp
```

Os resultados de entrada vêm principalmente de:

```text
results/metrics/full/
results/pairwise_metrics/full/
```

E os resultados deste notebook serão salvos em:

```text
results/statistical_analysis/full/
logs/statistical_analysis/
manifests/statistical_analysis/
```


In [1]:
import json
import math
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/workspace/pi-defense-exp")
EXPECTED_KERNEL = "Python (pi-defense-exp)"
EXPECTED_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"

print("Python executable:", sys.executable)
print("Expected Python:", EXPECTED_PYTHON)

if Path(sys.executable) != EXPECTED_PYTHON:
    print(
        "Aviso: o Python atual não é exatamente o Python esperado do projeto. "
        "Confirme se o kernel ativo é Python (pi-defense-exp)."
    )


Python executable: /workspace/pi-defense-exp/.venv/bin/python
Expected Python: /workspace/pi-defense-exp/.venv/bin/python


In [2]:
RESULTS_DIR = PROJECT_ROOT / "results"
LOG_DIR = PROJECT_ROOT / "logs" / "statistical_analysis"
MANIFEST_DIR = PROJECT_ROOT / "manifests" / "statistical_analysis"

ANALYSIS_RUN_MODE = "full"  # opções esperadas: "full" ou "smoke"

if ANALYSIS_RUN_MODE not in {"full", "smoke"}:
    raise ValueError("ANALYSIS_RUN_MODE deve ser 'full' ou 'smoke'.")

METRICS_06_DIR = RESULTS_DIR / "metrics" / ANALYSIS_RUN_MODE
PAIRWISE_07_DIR = RESULTS_DIR / "pairwise_metrics" / ANALYSIS_RUN_MODE
ANALYSIS_RESULTS_DIR = RESULTS_DIR / "statistical_analysis" / ANALYSIS_RUN_MODE

METRICS_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "metrics" / "06_compute_metrics_manifest.json"
PAIRWISE_MANIFEST_PATH = PROJECT_ROOT / "manifests" / "pairwise_metrics" / "07_pairwise_and_injected_metrics_manifest.json"

for path in [LOG_DIR, MANIFEST_DIR, ANALYSIS_RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

EVENTS_LOG_PATH = LOG_DIR / "08_statistical_analysis_events.jsonl"
SUMMARY_LOG_PATH = LOG_DIR / f"08_statistical_analysis_summary_{ANALYSIS_RUN_MODE}.json"

print("Analysis run mode:", ANALYSIS_RUN_MODE)
print("Metrics 06 dir:", METRICS_06_DIR)
print("Pairwise 07 dir:", PAIRWISE_07_DIR)
print("Analysis results dir:", ANALYSIS_RESULTS_DIR)
print("Events log path:", EVENTS_LOG_PATH)


Analysis run mode: full
Metrics 06 dir: /workspace/pi-defense-exp/results/metrics/full
Pairwise 07 dir: /workspace/pi-defense-exp/results/pairwise_metrics/full
Analysis results dir: /workspace/pi-defense-exp/results/statistical_analysis/full
Events log path: /workspace/pi-defense-exp/logs/statistical_analysis/08_statistical_analysis_events.jsonl


## 3. Funções utilitárias

As funções abaixo padronizam leitura e escrita de arquivos, registro incremental de eventos e geração de tabelas Markdown sem depender de pacotes opcionais.

Também há uma função de sanitização de JSON. Ela evita salvar valores inválidos como `NaN` cru em arquivos `.json`, o que pode quebrar renderizadores e parsers mais estritos.


In [3]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sanitize_json_value(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): sanitize_json_value(item) for key, item in value.items()}

    if isinstance(value, (list, tuple, set)):
        return [sanitize_json_value(item) for item in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating, float)):
        value = float(value)
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return value

    if isinstance(value, (np.bool_,)):
        return bool(value)

    try:
        if pd.isna(value):
            return "NaN"
    except (TypeError, ValueError):
        pass

    return value


def read_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            sanitize_json_value(data),
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
            allow_nan=False,
        )


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(sanitize_json_value(row), ensure_ascii=False, default=str, allow_nan=False) + "\n")


def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def log_event(event_type: str, payload: dict | None = None) -> None:
    event = {
        "timestamp_utc": utc_now(),
        "event_type": event_type,
    }
    if payload:
        event.update(payload)
    write_jsonl(EVENTS_LOG_PATH, [event]) if not EVENTS_LOG_PATH.exists() else append_jsonl(EVENTS_LOG_PATH, event)


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(sanitize_json_value(row), ensure_ascii=False, default=str, allow_nan=False) + "\n")


def dataframe_to_markdown_table(df: pd.DataFrame, max_rows: int | None = None) -> str:
    if df.empty:
        return "_Tabela vazia._"

    table_df = df.copy()
    if max_rows is not None:
        table_df = table_df.head(max_rows)

    columns = [str(column) for column in table_df.columns]
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]

    for _, row in table_df.iterrows():
        values = []
        for column in table_df.columns:
            value = row[column]
            if pd.isna(value):
                rendered = ""
            elif isinstance(value, float):
                rendered = f"{value:.6f}"
            else:
                rendered = str(value)
            rendered = rendered.replace("|", "\\|")
            values.append(rendered)
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)


log_event("statistical_analysis_started", {"run_mode": ANALYSIS_RUN_MODE})


## 4. Carregar manifestos dos notebooks anteriores

Os manifestos ajudam a rastrear quais arquivos foram produzidos e com quais configurações. Este notebook usa os manifestos como documentação de entrada, mas também valida diretamente a existência dos arquivos principais.

O manifesto do `06` é obrigatório, porque contém as métricas diretas principais. O manifesto do `07` é desejável, mas pode ainda não existir se as métricas pareadas não tiverem sido executadas. Nesse caso, o notebook continua e consolida apenas as métricas disponíveis.


In [4]:
metrics_manifest = None
pairwise_manifest = None

if not METRICS_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifesto do notebook 06 não encontrado: {METRICS_MANIFEST_PATH}. "
        "Execute o notebook 06 antes do 08."
    )

metrics_manifest = read_json(METRICS_MANIFEST_PATH)
print("Manifesto 06 carregado:", METRICS_MANIFEST_PATH)
print("Run mode 06:", metrics_manifest.get("run_mode"))

if PAIRWISE_MANIFEST_PATH.exists():
    pairwise_manifest = read_json(PAIRWISE_MANIFEST_PATH)
    print("Manifesto 07 carregado:", PAIRWISE_MANIFEST_PATH)
    print("Run mode 07:", pairwise_manifest.get("run_mode"))
else:
    print("Manifesto 07 não encontrado. O notebook seguirá apenas com as métricas disponíveis do 06.")

if metrics_manifest.get("run_mode") != ANALYSIS_RUN_MODE:
    print(
        "Aviso: o run_mode registrado no manifesto 06 difere do ANALYSIS_RUN_MODE. "
        "O notebook continuará usando os diretórios configurados neste notebook."
    )

if pairwise_manifest and pairwise_manifest.get("run_mode") != ANALYSIS_RUN_MODE:
    print(
        "Aviso: o run_mode registrado no manifesto 07 difere do ANALYSIS_RUN_MODE. "
        "O notebook continuará usando os diretórios configurados neste notebook."
    )


Manifesto 06 carregado: /workspace/pi-defense-exp/manifests/metrics/06_compute_metrics_manifest.json
Run mode 06: full
Manifesto 07 carregado: /workspace/pi-defense-exp/manifests/pairwise_metrics/07_pairwise_and_injected_metrics_manifest.json
Run mode 07: full


## 5. Carregar métricas diretas do notebook 06

O notebook `06_compute_metrics` produz métricas diretamente computáveis a partir de `expected_answer`, `attack_target` e `normalized_output`.

Os principais arquivos usados aqui são:

```text
compact_run_metrics.csv
compact_scenario_metrics.csv
run_level_metrics.csv
scenario_level_metrics.csv
```

A tabela por run é a mais importante para esta etapa, porque preserva a granularidade por cenário e seed. É a partir dela que calculamos média, desvio padrão e intervalos de confiança por cenário.


In [5]:
DIRECT_RUN_METRICS_PATH = METRICS_06_DIR / "compact_run_metrics.csv"
DIRECT_SCENARIO_METRICS_PATH = METRICS_06_DIR / "compact_scenario_metrics.csv"
RUN_LEVEL_METRICS_PATH = METRICS_06_DIR / "run_level_metrics.csv"
SCENARIO_LEVEL_METRICS_PATH = METRICS_06_DIR / "scenario_level_metrics.csv"

required_06_files = [DIRECT_RUN_METRICS_PATH]
for path in required_06_files:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo obrigatório do notebook 06 não encontrado: {path}")

direct_run_metrics_df = pd.read_csv(DIRECT_RUN_METRICS_PATH)
print("Direct run metrics:", DIRECT_RUN_METRICS_PATH)
print("Rows:", len(direct_run_metrics_df))
display(direct_run_metrics_df.head())

if DIRECT_SCENARIO_METRICS_PATH.exists():
    direct_scenario_metrics_df = pd.read_csv(DIRECT_SCENARIO_METRICS_PATH)
else:
    direct_scenario_metrics_df = pd.DataFrame()

if RUN_LEVEL_METRICS_PATH.exists():
    run_level_metrics_df = pd.read_csv(RUN_LEVEL_METRICS_PATH)
else:
    run_level_metrics_df = pd.DataFrame()

if SCENARIO_LEVEL_METRICS_PATH.exists():
    scenario_level_metrics_df = pd.read_csv(SCENARIO_LEVEL_METRICS_PATH)
else:
    scenario_level_metrics_df = pd.DataFrame()

log_event("metrics_06_loaded", {
    "direct_run_metrics_rows": int(len(direct_run_metrics_df)),
    "direct_run_metrics_path": str(DIRECT_RUN_METRICS_PATH),
})


Direct run metrics: /workspace/pi-defense-exp/results/metrics/full/compact_run_metrics.csv
Rows: 11


,scenario_id,seed,clean_accuracy,benign_utility,pna_t,clean_effectiveness,utility_drop,utility_under_attack_seen,robust_accuracy_seen,untargeted_asr_seen,...,untargeted_asr_unseen,targeted_asr_unseen,attack_success_rate_unseen,utility_under_attack_all_attacks,robust_accuracy_all_attacks,untargeted_asr_all_attacks,targeted_asr_all_attacks,attack_success_rate_all_attacks,robust_accuracy_delta_vs_c0_all_attacks,invalid_output_rate_all_attacks
0,c0_base,42,0.772921,0.772921,0.772921,1.000000,0.000000,0.123134,0.123134,0.876866,...,0.637527,0.630952,0.630952,0.212886,0.212886,0.787114,0.781850,0.781850,0.000000,0.000067
1,c1_struq_format_only,42,0.773454,0.773454,0.773454,1.000690,-0.000533,0.154264,0.154264,0.845736,...,0.697761,0.692786,0.692786,0.209755,0.209755,0.790245,0.785181,0.785181,-0.003132,0.000400
2,c2_struq_sft,42,0.859275,0.859275,0.859275,1.111724,-0.086354,0.986994,0.986994,0.013006,...,0.017768,0.005864,0.005864,0.985208,0.985208,0.014792,0.002665,0.002665,0.772321,0.000000
3,c2_struq_sft,123,0.855544,0.855544,0.855544,1.106897,-0.082623,0.985608,0.985608,0.014392,...,0.022566,0.011194,0.011194,0.982543,0.982543,0.017457,0.005330,0.005330,0.769656,0.000133
4,c2_struq_sft,2026,0.856077,0.856077,0.856077,1.107586,-0.083156,0.986141,0.986141,0.013859,...,0.021322,0.009773,0.009773,0.983342,0.983342,0.016658,0.004931,0.004931,0.770456,0.000000


## 6. Carregar métricas pareadas e injected-only do notebook 07

O notebook `07_pairwise_and_injected_metrics` produz métricas que exigem uma etapa adicional em relação ao `06`:

```text
PNA-I
MR
MR targeted
Win Rate
Adjusted Win Rate
```

Essas métricas são carregadas se os arquivos estiverem disponíveis. Caso o `07` ainda não tenha sido executado completamente, este notebook continua funcionando com as métricas do `06` e registra no manifesto quais métricas do `07` estavam ausentes.


In [6]:
PAIRWISE_COMPACT_PATH = PAIRWISE_07_DIR / "compact_07_metrics.csv"
PNA_I_RUN_METRICS_PATH = PAIRWISE_07_DIR / "pna_i_run_metrics.csv"
MR_RUN_METRICS_PATH = PAIRWISE_07_DIR / "mr_run_metrics.csv"
WIN_RATE_METRICS_PATH = PAIRWISE_07_DIR / "win_rate_metrics.csv"

pairwise_compact_df = pd.read_csv(PAIRWISE_COMPACT_PATH) if PAIRWISE_COMPACT_PATH.exists() else pd.DataFrame()
pna_i_run_metrics_df = pd.read_csv(PNA_I_RUN_METRICS_PATH) if PNA_I_RUN_METRICS_PATH.exists() else pd.DataFrame()
mr_run_metrics_df = pd.read_csv(MR_RUN_METRICS_PATH) if MR_RUN_METRICS_PATH.exists() else pd.DataFrame()
win_rate_metrics_df = pd.read_csv(WIN_RATE_METRICS_PATH) if WIN_RATE_METRICS_PATH.exists() else pd.DataFrame()

print("Pairwise compact exists:", PAIRWISE_COMPACT_PATH.exists(), PAIRWISE_COMPACT_PATH)
print("PNA-I run metrics exists:", PNA_I_RUN_METRICS_PATH.exists(), PNA_I_RUN_METRICS_PATH)
print("MR run metrics exists:", MR_RUN_METRICS_PATH.exists(), MR_RUN_METRICS_PATH)
print("Win Rate metrics exists:", WIN_RATE_METRICS_PATH.exists(), WIN_RATE_METRICS_PATH)

if not pairwise_compact_df.empty:
    print("Pairwise compact rows:", len(pairwise_compact_df))
    display(pairwise_compact_df.head())
else:
    print("Métricas compactas do 07 não encontradas. As análises pareadas serão puladas.")

log_event("metrics_07_loaded", {
    "pairwise_compact_available": bool(not pairwise_compact_df.empty),
    "pairwise_compact_rows": int(len(pairwise_compact_df)),
    "pna_i_rows": int(len(pna_i_run_metrics_df)),
    "mr_rows": int(len(mr_run_metrics_df)),
    "win_rate_rows": int(len(win_rate_metrics_df)),
})


Pairwise compact exists: True /workspace/pi-defense-exp/results/pairwise_metrics/full/compact_07_metrics.csv
PNA-I run metrics exists: True /workspace/pi-defense-exp/results/pairwise_metrics/full/pna_i_run_metrics.csv
MR run metrics exists: True /workspace/pi-defense-exp/results/pairwise_metrics/full/mr_run_metrics.csv
Win Rate metrics exists: True /workspace/pi-defense-exp/results/pairwise_metrics/full/win_rate_metrics.csv
Pairwise compact rows: 11


,scenario_id,scenario_label,seed,pna_i_seen,pna_i_unseen,mr_seen,mr_targeted_seen,mr_unseen,mr_targeted_unseen,win_rate_clean_vs_c0,tie_rate_clean_vs_c0,adjusted_win_rate_clean_vs_c0
0,c0_base,C0 — Base model,42,0.926119,0.897299,0.810341,0.805011,0.630419,0.587420,NaN,NaN,NaN
1,c1_struq_format_only,C1 — StruQ format-only,42,0.725267,0.797974,0.595096,0.585501,0.637704,0.573738,0.003333,0.920000,0.463333
2,c2_struq_sft,C2 — StruQ-like SFT,42,0.000000,0.019367,0.988166,0.000000,0.945807,0.000000,0.010000,0.886667,0.453333
3,c2_struq_sft,C2 — StruQ-like SFT,123,0.000000,0.019367,0.875693,0.000000,0.867804,0.000000,0.006667,0.860000,0.436667
4,c2_struq_sft,C2 — StruQ-like SFT,2026,0.000000,0.036958,0.877079,0.000000,0.852523,0.000178,0.013378,0.862876,0.444816


## 7. Construir tabela unificada por run

Nesta etapa, as métricas do `06` e do `07` são combinadas em uma única tabela por cenário e seed.

Essa tabela é a base da análise estatística deste notebook. Cada linha representa uma execução experimental de um cenário com uma seed específica.

Para os cenários sem treinamento (`C0` e `C1`), normalmente haverá apenas a seed `42`. Para os cenários treinados (`C2`, `C3` e `C4`), esperamos três seeds:

```text
42, 123, 2026
```


In [7]:
SCENARIO_LABELS = {
    "c0_base": "C0 — Base model",
    "c1_struq_format_only": "C1 — StruQ format-only",
    "c2_struq_sft": "C2 — StruQ-like SFT",
    "c3_secalign_dpo": "C3 — SecAlign-like DPO",
    "c4_ih_sft": "C4 — Instruction-Hierarchy-like SFT",
}

SCENARIO_ORDER = [
    "c0_base",
    "c1_struq_format_only",
    "c2_struq_sft",
    "c3_secalign_dpo",
    "c4_ih_sft",
]


def add_scenario_metadata(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    if "scenario_label" not in df.columns:
        df["scenario_label"] = df["scenario_id"].map(SCENARIO_LABELS)
    else:
        df["scenario_label"] = df["scenario_label"].fillna(df["scenario_id"].map(SCENARIO_LABELS))
    df["scenario_order"] = df["scenario_id"].map({scenario: idx for idx, scenario in enumerate(SCENARIO_ORDER)})
    return df


direct_run_metrics_df = add_scenario_metadata(direct_run_metrics_df)
pairwise_compact_df = add_scenario_metadata(pairwise_compact_df)

if not pairwise_compact_df.empty:
    columns_to_drop = [
        column
        for column in ["scenario_label", "scenario_order"]
        if column in pairwise_compact_df.columns
    ]
    pairwise_for_merge = pairwise_compact_df.drop(columns=columns_to_drop)
    unified_run_metrics_df = direct_run_metrics_df.merge(
        pairwise_for_merge,
        on=["scenario_id", "seed"],
        how="outer",
        suffixes=("", "_07"),
    )
else:
    unified_run_metrics_df = direct_run_metrics_df.copy()

unified_run_metrics_df = add_scenario_metadata(unified_run_metrics_df)
unified_run_metrics_df = unified_run_metrics_df.sort_values(["scenario_order", "seed"]).reset_index(drop=True)

unified_run_metrics_path = ANALYSIS_RESULTS_DIR / "unified_run_metrics.csv"
unified_run_metrics_jsonl_path = ANALYSIS_RESULTS_DIR / "unified_run_metrics.jsonl"
unified_run_metrics_df.to_csv(unified_run_metrics_path, index=False)
write_jsonl(unified_run_metrics_jsonl_path, unified_run_metrics_df.to_dict(orient="records"))

print("Tabela unificada por run salva em:", unified_run_metrics_path)
print("Rows:", len(unified_run_metrics_df))
display(unified_run_metrics_df.head(10))


Tabela unificada por run salva em: /workspace/pi-defense-exp/results/statistical_analysis/full/unified_run_metrics.csv
Rows: 11


,scenario_id,seed,clean_accuracy,benign_utility,pna_t,clean_effectiveness,utility_drop,utility_under_attack_seen,robust_accuracy_seen,untargeted_asr_seen,...,scenario_order,pna_i_seen,pna_i_unseen,mr_seen,mr_targeted_seen,mr_unseen,mr_targeted_unseen,win_rate_clean_vs_c0,tie_rate_clean_vs_c0,adjusted_win_rate_clean_vs_c0
0,c0_base,42,0.772921,0.772921,0.772921,1.000000,0.000000,0.123134,0.123134,0.876866,...,0,0.926119,0.897299,0.810341,0.805011,0.630419,0.587420,NaN,NaN,NaN
1,c1_struq_format_only,42,0.773454,0.773454,0.773454,1.000690,-0.000533,0.154264,0.154264,0.845736,...,1,0.725267,0.797974,0.595096,0.585501,0.637704,0.573738,0.003333,0.920000,0.463333
2,c2_struq_sft,42,0.859275,0.859275,0.859275,1.111724,-0.086354,0.986994,0.986994,0.013006,...,2,0.000000,0.019367,0.988166,0.000000,0.945807,0.000000,0.010000,0.886667,0.453333
3,c2_struq_sft,123,0.855544,0.855544,0.855544,1.106897,-0.082623,0.985608,0.985608,0.014392,...,2,0.000000,0.019367,0.875693,0.000000,0.867804,0.000000,0.006667,0.860000,0.436667
4,c2_struq_sft,2026,0.856077,0.856077,0.856077,1.107586,-0.083156,0.986141,0.986141,0.013859,...,2,0.000000,0.036958,0.877079,0.000000,0.852523,0.000178,0.013378,0.862876,0.444816
5,c3_secalign_dpo,42,0.746269,0.746269,0.746269,0.965517,0.026652,0.963326,0.963326,0.036674,...,3,0.226759,0.445984,0.707463,0.009808,0.504264,0.028429,0.006667,0.863333,0.438333
6,c3_secalign_dpo,123,0.728145,0.728145,0.728145,0.942069,0.044776,0.964286,0.964286,0.035714,...,3,0.157783,0.150320,0.792431,0.005650,0.770256,0.010128,0.003333,0.840000,0.423333
7,c3_secalign_dpo,2026,0.778252,0.778252,0.778252,1.006897,-0.005330,0.806290,0.806290,0.193710,...,3,0.504904,0.636638,0.472601,0.114499,0.393746,0.118159,0.006667,0.826667,0.420000
8,c4_ih_sft,42,0.858209,0.858209,0.858209,1.110345,-0.085288,0.986141,0.986141,0.013859,...,4,0.000000,0.000000,0.991151,0.000000,0.988984,0.000000,0.006667,0.870000,0.441667
9,c4_ih_sft,123,0.860874,0.860874,0.860874,1.113793,-0.087953,0.985608,0.985608,0.014392,...,4,0.000000,0.000000,0.988913,0.000000,0.988095,0.000000,0.006667,0.876667,0.445000


## 8. Catálogo de métricas e direção interpretativa

Nem toda métrica deve ser interpretada da mesma forma.

Algumas métricas são melhores quando são maiores, como:

```text
clean_accuracy
robust_accuracy_all_attacks
adjusted_win_rate_clean_vs_c0
```

Outras são melhores quando são menores, como:

```text
utility_drop
targeted_asr_all_attacks
mr_targeted_seen
invalid_output_rate_all_attacks
```

Também existem métricas diagnósticas, como `pna_i`, que ajudam a entender o comportamento do modelo, mas não devem ser automaticamente tratadas como “maior é melhor” ou “menor é melhor” sem explicar o contexto.

Essa distinção será usada para rankings e deltas ajustados por direção.


In [8]:
METRIC_DIRECTIONS = {
    # Utility / clean behavior
    "clean_accuracy": "higher",
    "benign_utility": "higher",
    "pna_t": "higher",
    "clean_effectiveness": "higher",
    "utility_drop": "lower",

    # Robustness under attack
    "utility_under_attack_seen": "higher",
    "utility_under_attack_unseen": "higher",
    "utility_under_attack_all_attacks": "higher",
    "robust_accuracy_seen": "higher",
    "robust_accuracy_unseen": "higher",
    "robust_accuracy_all_attacks": "higher",
    "robust_accuracy_delta_vs_c0_all_attacks": "higher",

    # Attack success / injection following
    "untargeted_asr_seen": "lower",
    "untargeted_asr_unseen": "lower",
    "untargeted_asr_all_attacks": "lower",
    "targeted_asr_seen": "lower",
    "targeted_asr_unseen": "lower",
    "targeted_asr_all_attacks": "lower",
    "attack_success_rate_seen": "lower",
    "attack_success_rate_unseen": "lower",
    "attack_success_rate_all_attacks": "lower",
    "injection_following_rate_seen": "lower",
    "injection_following_rate_unseen": "lower",
    "injection_following_rate_all_attacks": "lower",
    "binary_asv_seen": "lower",
    "binary_asv_unseen": "lower",
    "binary_asv_all_attacks": "lower",

    # Output validity
    "invalid_output_rate_all_attacks": "lower",
    "valid_output_rate_all_attacks": "higher",

    # Injected-only and match rate
    "pna_i_seen": "neutral",
    "pna_i_unseen": "neutral",
    "mr_seen": "lower",
    "mr_unseen": "lower",
    "mr_targeted_seen": "lower",
    "mr_targeted_unseen": "lower",

    # Win Rate
    "win_rate_clean_vs_c0": "higher",
    "tie_rate_clean_vs_c0": "neutral",
    "adjusted_win_rate_clean_vs_c0": "higher",
}

PREFERRED_REPORT_METRICS = [
    "clean_accuracy",
    "utility_drop",
    "robust_accuracy_seen",
    "robust_accuracy_unseen",
    "robust_accuracy_all_attacks",
    "targeted_asr_seen",
    "targeted_asr_unseen",
    "targeted_asr_all_attacks",
    "untargeted_asr_all_attacks",
    "invalid_output_rate_all_attacks",
    "pna_i_seen",
    "pna_i_unseen",
    "mr_seen",
    "mr_unseen",
    "mr_targeted_seen",
    "mr_targeted_unseen",
    "adjusted_win_rate_clean_vs_c0",
]

available_report_metrics = [
    metric for metric in PREFERRED_REPORT_METRICS
    if metric in unified_run_metrics_df.columns
]

metric_catalog_rows = []
for metric in available_report_metrics:
    metric_catalog_rows.append({
        "metric": metric,
        "direction": METRIC_DIRECTIONS.get(metric, "unknown"),
        "available": metric in unified_run_metrics_df.columns,
    })

metric_catalog_df = pd.DataFrame(metric_catalog_rows)
metric_catalog_path = ANALYSIS_RESULTS_DIR / "metric_catalog_08.csv"
metric_catalog_df.to_csv(metric_catalog_path, index=False)

display(metric_catalog_df)


,metric,direction,available
0,clean_accuracy,higher,True
1,utility_drop,lower,True
2,robust_accuracy_seen,higher,True
3,robust_accuracy_unseen,higher,True
4,robust_accuracy_all_attacks,higher,True
5,targeted_asr_seen,lower,True
6,targeted_asr_unseen,lower,True
7,targeted_asr_all_attacks,lower,True
8,untargeted_asr_all_attacks,lower,True
9,invalid_output_rate_all_attacks,lower,True


## 9. Resumo estatístico por cenário

Agora calculamos estatísticas por cenário para cada métrica disponível.

Para cada cenário e métrica, serão registrados:

```text
n
mean
std
sem
ci95_low
ci95_high
min
max
```

O intervalo de confiança é calculado a partir das seeds disponíveis. Quando há apenas uma seed, o desvio padrão e o intervalo de confiança ficam ausentes, porque não há variação amostral suficiente para estimá-los.

Como temos apenas três seeds para os cenários treinados, estes intervalos devem ser interpretados como uma descrição da variabilidade observada, não como inferência estatística forte.


In [9]:
def get_t_critical_95(n: int) -> float:
    if n <= 1:
        return float("nan")
    try:
        from scipy.stats import t
        return float(t.ppf(0.975, df=n - 1))
    except Exception:
        # Valores comuns para n pequeno; fallback para normal quando necessário.
        t_values = {
            2: 12.706,
            3: 4.303,
            4: 3.182,
            5: 2.776,
            6: 2.571,
            7: 2.447,
            8: 2.365,
            9: 2.306,
            10: 2.262,
        }
        return t_values.get(n, 1.96)


def summarize_metric_series(values: pd.Series) -> dict:
    numeric_values = pd.to_numeric(values, errors="coerce").dropna()
    n = int(len(numeric_values))

    if n == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "std": np.nan,
            "sem": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "min": np.nan,
            "max": np.nan,
        }

    mean = float(numeric_values.mean())
    std = float(numeric_values.std(ddof=1)) if n > 1 else np.nan
    sem = float(std / math.sqrt(n)) if n > 1 else np.nan
    tcrit = get_t_critical_95(n)
    ci_half_width = tcrit * sem if n > 1 else np.nan

    return {
        "n": n,
        "mean": mean,
        "std": std,
        "sem": sem,
        "ci95_low": mean - ci_half_width if n > 1 else np.nan,
        "ci95_high": mean + ci_half_width if n > 1 else np.nan,
        "min": float(numeric_values.min()),
        "max": float(numeric_values.max()),
    }


summary_rows = []
for scenario_id, scenario_df in unified_run_metrics_df.groupby("scenario_id", dropna=False):
    scenario_label = scenario_df["scenario_label"].dropna().iloc[0] if scenario_df["scenario_label"].notna().any() else scenario_id
    scenario_order = scenario_df["scenario_order"].dropna().iloc[0] if scenario_df["scenario_order"].notna().any() else 999

    for metric in available_report_metrics:
        stats = summarize_metric_series(scenario_df[metric])
        summary_rows.append({
            "scenario_id": scenario_id,
            "scenario_label": scenario_label,
            "scenario_order": int(scenario_order),
            "metric": metric,
            "direction": METRIC_DIRECTIONS.get(metric, "unknown"),
            **stats,
        })

scenario_metric_summary_df = pd.DataFrame(summary_rows).sort_values(["scenario_order", "metric"])
scenario_metric_summary_path = ANALYSIS_RESULTS_DIR / "scenario_metric_summary.csv"
scenario_metric_summary_jsonl_path = ANALYSIS_RESULTS_DIR / "scenario_metric_summary.jsonl"
scenario_metric_summary_df.to_csv(scenario_metric_summary_path, index=False)
write_jsonl(scenario_metric_summary_jsonl_path, scenario_metric_summary_df.to_dict(orient="records"))

print("Resumo por cenário/métrica salvo em:", scenario_metric_summary_path)
display(scenario_metric_summary_df.head(20))


Resumo por cenário/métrica salvo em: /workspace/pi-defense-exp/results/statistical_analysis/full/scenario_metric_summary.csv


,scenario_id,scenario_label,scenario_order,metric,direction,n,mean,std,sem,ci95_low,ci95_high,min,max
16,c0_base,C0 — Base model,0,adjusted_win_rate_clean_vs_c0,higher,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,c0_base,C0 — Base model,0,clean_accuracy,higher,1,0.772921,NaN,NaN,NaN,NaN,0.772921,0.772921
9,c0_base,C0 — Base model,0,invalid_output_rate_all_attacks,lower,1,0.000067,NaN,NaN,NaN,NaN,0.000067,0.000067
12,c0_base,C0 — Base model,0,mr_seen,lower,1,0.810341,NaN,NaN,NaN,NaN,0.810341,0.810341
14,c0_base,C0 — Base model,0,mr_targeted_seen,lower,1,0.805011,NaN,NaN,NaN,NaN,0.805011,0.805011
15,c0_base,C0 — Base model,0,mr_targeted_unseen,lower,1,0.587420,NaN,NaN,NaN,NaN,0.587420,0.587420
13,c0_base,C0 — Base model,0,mr_unseen,lower,1,0.630419,NaN,NaN,NaN,NaN,0.630419,0.630419
10,c0_base,C0 — Base model,0,pna_i_seen,neutral,1,0.926119,NaN,NaN,NaN,NaN,0.926119,0.926119
11,c0_base,C0 — Base model,0,pna_i_unseen,neutral,1,0.897299,NaN,NaN,NaN,NaN,0.897299,0.897299
4,c0_base,C0 — Base model,0,robust_accuracy_all_attacks,higher,1,0.212886,NaN,NaN,NaN,NaN,0.212886,0.212886


## 10. Tabela compacta para leitura rápida

A tabela anterior é longa, pois contém uma linha por cenário e métrica. Agora criamos uma versão compacta, com uma linha por cenário e colunas no formato:

```text
<metric>_mean
<metric>_std
```

Essa tabela é útil para uma visão geral rápida e também para exportação posterior.


In [10]:
compact_rows = []
for scenario_id, scenario_df in unified_run_metrics_df.groupby("scenario_id", dropna=False):
    scenario_label = scenario_df["scenario_label"].dropna().iloc[0] if scenario_df["scenario_label"].notna().any() else scenario_id
    scenario_order = scenario_df["scenario_order"].dropna().iloc[0] if scenario_df["scenario_order"].notna().any() else 999

    row = {
        "scenario_id": scenario_id,
        "scenario_label": scenario_label,
        "scenario_order": int(scenario_order),
        "n_runs": int(scenario_df["seed"].nunique()),
        "seeds": ",".join(str(seed) for seed in sorted(scenario_df["seed"].dropna().astype(int).unique())),
    }

    for metric in available_report_metrics:
        stats = summarize_metric_series(scenario_df[metric])
        row[f"{metric}_mean"] = stats["mean"]
        row[f"{metric}_std"] = stats["std"]
        row[f"{metric}_ci95_low"] = stats["ci95_low"]
        row[f"{metric}_ci95_high"] = stats["ci95_high"]

    compact_rows.append(row)

scenario_compact_summary_df = pd.DataFrame(compact_rows).sort_values("scenario_order")
scenario_compact_summary_path = ANALYSIS_RESULTS_DIR / "scenario_compact_summary.csv"
scenario_compact_summary_jsonl_path = ANALYSIS_RESULTS_DIR / "scenario_compact_summary.jsonl"
scenario_compact_summary_df.to_csv(scenario_compact_summary_path, index=False)
write_jsonl(scenario_compact_summary_jsonl_path, scenario_compact_summary_df.to_dict(orient="records"))

print("Tabela compacta por cenário salva em:", scenario_compact_summary_path)
display(scenario_compact_summary_df)


Tabela compacta por cenário salva em: /workspace/pi-defense-exp/results/statistical_analysis/full/scenario_compact_summary.csv


,scenario_id,scenario_label,scenario_order,n_runs,seeds,clean_accuracy_mean,clean_accuracy_std,clean_accuracy_ci95_low,clean_accuracy_ci95_high,utility_drop_mean,...,mr_targeted_seen_ci95_low,mr_targeted_seen_ci95_high,mr_targeted_unseen_mean,mr_targeted_unseen_std,mr_targeted_unseen_ci95_low,mr_targeted_unseen_ci95_high,adjusted_win_rate_clean_vs_c0_mean,adjusted_win_rate_clean_vs_c0_std,adjusted_win_rate_clean_vs_c0_ci95_low,adjusted_win_rate_clean_vs_c0_ci95_high
0,c0_base,C0 — Base model,0,1,42,0.772921,NaN,NaN,NaN,0.000000,...,NaN,NaN,0.587420,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,c1_struq_format_only,C1 — StruQ format-only,1,1,42,0.773454,NaN,NaN,NaN,-0.000533,...,NaN,NaN,0.573738,NaN,NaN,NaN,0.463333,NaN,NaN,NaN
2,c2_struq_sft,C2 — StruQ-like SFT,2,3,"42,123,2026",0.856965,0.002018,0.851952,0.861978,-0.084044,...,0.000000,0.000000,0.000059,0.000103,-0.000196,0.000314,0.444939,0.008334,0.424236,0.465642
3,c3_secalign_dpo,C3 — SecAlign-like DPO,3,3,"42,123,2026",0.750888,0.025371,0.687864,0.813913,0.022033,...,-0.109899,0.196537,0.052239,0.057817,-0.091388,0.195865,0.427222,0.009766,0.402963,0.451482
4,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,4,3,"42,123,2026",0.856432,0.005548,0.842650,0.870214,-0.083511,...,0.000000,0.000000,0.000296,0.000513,-0.000978,0.001570,0.442778,0.001925,0.437997,0.447559


## 11. Deltas em relação ao C0 e ao C1

Além das médias absolutas, é útil calcular a diferença em relação a cenários de referência.

Neste notebook usamos duas referências:

```text
C0 — Base model
C1 — StruQ format-only
```

O delta bruto é calculado como:

```text
valor_do_cenario - valor_da_referencia
```

Além disso, calculamos um delta ajustado por direção interpretativa. Para métricas em que maior é melhor, o delta ajustado é igual ao delta bruto. Para métricas em que menor é melhor, o sinal é invertido. Assim, no delta ajustado:

```text
valor positivo = melhora em relação à referência
valor negativo = piora em relação à referência
```

Essa transformação facilita comparar métricas de natureza diferente, como `robust_accuracy` e `targeted_asr`.


In [11]:
def get_reference_value(summary_df: pd.DataFrame, reference_scenario_id: str, metric: str) -> float | None:
    match = summary_df[
        (summary_df["scenario_id"] == reference_scenario_id)
        & (summary_df["metric"] == metric)
    ]
    if match.empty:
        return None
    value = match.iloc[0]["mean"]
    if pd.isna(value):
        return None
    return float(value)


def direction_adjusted_delta(metric: str, raw_delta: float | None) -> float | None:
    if raw_delta is None or pd.isna(raw_delta):
        return None
    direction = METRIC_DIRECTIONS.get(metric, "unknown")
    if direction == "higher":
        return raw_delta
    if direction == "lower":
        return -raw_delta
    return None


reference_scenarios = ["c0_base", "c1_struq_format_only"]
delta_rows = []

for _, row in scenario_metric_summary_df.iterrows():
    scenario_id = row["scenario_id"]
    metric = row["metric"]
    scenario_value = row["mean"]

    for reference_scenario_id in reference_scenarios:
        if scenario_id == reference_scenario_id:
            continue

        reference_value = get_reference_value(
            scenario_metric_summary_df,
            reference_scenario_id=reference_scenario_id,
            metric=metric,
        )

        if reference_value is None or pd.isna(scenario_value):
            raw_delta = np.nan
            adjusted_delta = np.nan
        else:
            raw_delta = float(scenario_value) - float(reference_value)
            adjusted_delta = direction_adjusted_delta(metric, raw_delta)

        delta_rows.append({
            "scenario_id": scenario_id,
            "scenario_label": row["scenario_label"],
            "reference_scenario_id": reference_scenario_id,
            "reference_scenario_label": SCENARIO_LABELS.get(reference_scenario_id, reference_scenario_id),
            "metric": metric,
            "direction": METRIC_DIRECTIONS.get(metric, "unknown"),
            "scenario_mean": scenario_value,
            "reference_mean": reference_value,
            "raw_delta": raw_delta,
            "direction_adjusted_delta": adjusted_delta,
        })

deltas_vs_reference_df = pd.DataFrame(delta_rows)
deltas_vs_reference_path = ANALYSIS_RESULTS_DIR / "deltas_vs_reference.csv"
deltas_vs_reference_jsonl_path = ANALYSIS_RESULTS_DIR / "deltas_vs_reference.jsonl"
deltas_vs_reference_df.to_csv(deltas_vs_reference_path, index=False)
write_jsonl(deltas_vs_reference_jsonl_path, deltas_vs_reference_df.to_dict(orient="records"))

print("Deltas salvos em:", deltas_vs_reference_path)
display(deltas_vs_reference_df.head(30))


Deltas salvos em: /workspace/pi-defense-exp/results/statistical_analysis/full/deltas_vs_reference.csv


,scenario_id,scenario_label,reference_scenario_id,reference_scenario_label,metric,direction,scenario_mean,reference_mean,raw_delta,direction_adjusted_delta
0,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,adjusted_win_rate_clean_vs_c0,higher,NaN,0.463333,NaN,NaN
1,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,clean_accuracy,higher,0.772921,0.773454,-0.000533,-0.000533
2,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,invalid_output_rate_all_attacks,lower,0.000067,0.000400,-0.000333,0.000333
3,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,mr_seen,lower,0.810341,0.595096,0.215245,-0.215245
4,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,mr_targeted_seen,lower,0.805011,0.585501,0.219510,-0.219510
5,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,mr_targeted_unseen,lower,0.587420,0.573738,0.013682,-0.013682
6,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,mr_unseen,lower,0.630419,0.637704,-0.007285,0.007285
7,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,pna_i_seen,neutral,0.926119,0.725267,0.200853,NaN
8,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,pna_i_unseen,neutral,0.897299,0.797974,0.099325,NaN
9,c0_base,C0 — Base model,c1_struq_format_only,C1 — StruQ format-only,robust_accuracy_all_attacks,higher,0.212886,0.209755,0.003132,0.003132


## 12. Rankings por métrica

Agora produzimos rankings por métrica, respeitando a direção interpretativa de cada uma.

Para métricas em que maior é melhor, o maior valor recebe rank 1. Para métricas em que menor é melhor, o menor valor recebe rank 1.

Métricas classificadas como `neutral` não são ranqueadas automaticamente, porque sua interpretação depende do contexto metodológico. Esse é o caso de `pna_i`, que é útil para entender a força da instrução injetada isolada, mas não deve ser tratado diretamente como sucesso ou falha da defesa sem discussão.


In [12]:
rank_rows = []
for metric in available_report_metrics:
    direction = METRIC_DIRECTIONS.get(metric, "unknown")
    if direction not in {"higher", "lower"}:
        continue

    metric_df = scenario_metric_summary_df[
        scenario_metric_summary_df["metric"] == metric
    ].copy()
    metric_df = metric_df.dropna(subset=["mean"])

    if metric_df.empty:
        continue

    ascending = direction == "lower"
    metric_df["rank"] = metric_df["mean"].rank(method="min", ascending=ascending)

    for _, row in metric_df.sort_values("rank").iterrows():
        rank_rows.append({
            "metric": metric,
            "direction": direction,
            "rank": int(row["rank"]),
            "scenario_id": row["scenario_id"],
            "scenario_label": row["scenario_label"],
            "mean": row["mean"],
            "std": row["std"],
            "n": row["n"],
        })

ranking_df = pd.DataFrame(rank_rows)
ranking_path = ANALYSIS_RESULTS_DIR / "metric_rankings.csv"
ranking_jsonl_path = ANALYSIS_RESULTS_DIR / "metric_rankings.jsonl"
ranking_df.to_csv(ranking_path, index=False)
write_jsonl(ranking_jsonl_path, ranking_df.to_dict(orient="records"))

print("Rankings por métrica salvos em:", ranking_path)
display(ranking_df.head(40))


Rankings por métrica salvos em: /workspace/pi-defense-exp/results/statistical_analysis/full/metric_rankings.csv


,metric,direction,rank,scenario_id,scenario_label,mean,std,n
0,clean_accuracy,higher,1,c2_struq_sft,C2 — StruQ-like SFT,0.856965,0.002018,3
1,clean_accuracy,higher,2,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,0.856432,0.005548,3
2,clean_accuracy,higher,3,c1_struq_format_only,C1 — StruQ format-only,0.773454,NaN,1
3,clean_accuracy,higher,4,c0_base,C0 — Base model,0.772921,NaN,1
4,clean_accuracy,higher,5,c3_secalign_dpo,C3 — SecAlign-like DPO,0.750888,0.025371,3
5,utility_drop,lower,1,c2_struq_sft,C2 — StruQ-like SFT,-0.084044,0.002018,3
6,utility_drop,lower,2,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,-0.083511,0.005548,3
7,utility_drop,lower,3,c1_struq_format_only,C1 — StruQ format-only,-0.000533,NaN,1
8,utility_drop,lower,4,c0_base,C0 — Base model,0.000000,NaN,1
9,utility_drop,lower,5,c3_secalign_dpo,C3 — SecAlign-like DPO,0.022033,0.025371,3


## 13. Score exploratório de trade-off

Além das métricas individuais, pode ser útil criar uma visão agregada do trade-off entre utilidade e robustez.

Este score **não é uma métrica oficial dos artigos**. Ele é apenas uma ferramenta exploratória para ordenar cenários segundo um conjunto de critérios escolhido neste notebook.

A ideia é:

```text
- normalizar métricas para uma escala comum;
- inverter métricas em que menor é melhor;
- calcular uma média simples dos componentes disponíveis.
```

O score deve ser usado com cuidado. Ele não substitui as métricas originais e não deve ser apresentado como resultado principal sem explicar sua construção.


In [13]:
TRADEOFF_COMPONENTS = [
    "clean_accuracy",
    "robust_accuracy_all_attacks",
    "targeted_asr_all_attacks",
    "invalid_output_rate_all_attacks",
    "mr_targeted_seen",
    "mr_targeted_unseen",
    "adjusted_win_rate_clean_vs_c0",
]

available_tradeoff_components = [
    metric for metric in TRADEOFF_COMPONENTS
    if metric in available_report_metrics
]

tradeoff_base = scenario_metric_summary_df[
    scenario_metric_summary_df["metric"].isin(available_tradeoff_components)
].copy()

tradeoff_rows = []
for metric in available_tradeoff_components:
    metric_df = tradeoff_base[tradeoff_base["metric"] == metric].copy()
    values = metric_df["mean"].astype(float)

    min_value = values.min(skipna=True)
    max_value = values.max(skipna=True)
    direction = METRIC_DIRECTIONS.get(metric, "unknown")

    for _, row in metric_df.iterrows():
        value = row["mean"]
        if pd.isna(value) or pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
            normalized = np.nan
        else:
            if direction == "higher":
                normalized = (value - min_value) / (max_value - min_value)
            elif direction == "lower":
                normalized = (max_value - value) / (max_value - min_value)
            else:
                normalized = np.nan

        tradeoff_rows.append({
            "scenario_id": row["scenario_id"],
            "scenario_label": row["scenario_label"],
            "metric": metric,
            "direction": direction,
            "mean": value,
            "normalized_component": normalized,
        })

tradeoff_components_df = pd.DataFrame(tradeoff_rows)

if not tradeoff_components_df.empty:
    tradeoff_score_df = (
        tradeoff_components_df
        .groupby(["scenario_id", "scenario_label"], dropna=False)
        .agg(
            tradeoff_score=("normalized_component", "mean"),
            available_components=("normalized_component", lambda s: int(s.notna().sum())),
        )
        .reset_index()
    )
    tradeoff_score_df["rank"] = tradeoff_score_df["tradeoff_score"].rank(method="min", ascending=False)
    tradeoff_score_df = tradeoff_score_df.sort_values("rank")
else:
    tradeoff_score_df = pd.DataFrame()

tradeoff_components_path = ANALYSIS_RESULTS_DIR / "tradeoff_components.csv"
tradeoff_score_path = ANALYSIS_RESULTS_DIR / "tradeoff_score.csv"
tradeoff_components_df.to_csv(tradeoff_components_path, index=False)
tradeoff_score_df.to_csv(tradeoff_score_path, index=False)

write_jsonl(ANALYSIS_RESULTS_DIR / "tradeoff_components.jsonl", tradeoff_components_df.to_dict(orient="records"))
write_jsonl(ANALYSIS_RESULTS_DIR / "tradeoff_score.jsonl", tradeoff_score_df.to_dict(orient="records"))

print("Componentes usados no score exploratório:", available_tradeoff_components)
display(tradeoff_score_df)


Componentes usados no score exploratório: ['clean_accuracy', 'robust_accuracy_all_attacks', 'targeted_asr_all_attacks', 'invalid_output_rate_all_attacks', 'mr_targeted_seen', 'mr_targeted_unseen', 'adjusted_win_rate_clean_vs_c0']


,scenario_id,scenario_label,tradeoff_score,available_components,rank
2,c2_struq_sft,C2 — StruQ-like SFT,0.924824,7,1.0
4,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,0.917906,7,2.0
3,c3_secalign_dpo,C3 — SecAlign-like DPO,0.525359,7,3.0
1,c1_struq_format_only,C1 — StruQ format-only,0.346855,7,4.0
0,c0_base,C0 — Base model,0.200422,6,5.0


## 14. Estabilidade entre seeds

Para os cenários treinados, a variação entre seeds é importante. Uma defesa é mais confiável quando seu desempenho é relativamente estável entre execuções.

Nesta etapa, calculamos uma tabela de estabilidade por cenário e métrica. Ela inclui:

```text
mean
std
range
coefficient_of_variation
```

O coeficiente de variação é calculado como `std / |mean|`. Ele é útil quando a média não está próxima de zero. Quando a média é zero ou ausente, esse valor fica ausente.


In [14]:
stability_rows = []
for _, row in scenario_metric_summary_df.iterrows():
    mean = row["mean"]
    std = row["std"]
    min_value = row["min"]
    max_value = row["max"]

    if pd.isna(mean) or pd.isna(std) or mean == 0:
        coefficient_of_variation = np.nan
    else:
        coefficient_of_variation = abs(std / mean)

    if pd.isna(min_value) or pd.isna(max_value):
        value_range = np.nan
    else:
        value_range = max_value - min_value

    stability_rows.append({
        "scenario_id": row["scenario_id"],
        "scenario_label": row["scenario_label"],
        "metric": row["metric"],
        "direction": row["direction"],
        "n": row["n"],
        "mean": mean,
        "std": std,
        "range": value_range,
        "coefficient_of_variation": coefficient_of_variation,
    })

stability_df = pd.DataFrame(stability_rows)
stability_path = ANALYSIS_RESULTS_DIR / "seed_stability.csv"
stability_jsonl_path = ANALYSIS_RESULTS_DIR / "seed_stability.jsonl"
stability_df.to_csv(stability_path, index=False)
write_jsonl(stability_jsonl_path, stability_df.to_dict(orient="records"))

print("Tabela de estabilidade salva em:", stability_path)
display(stability_df.head(30))


Tabela de estabilidade salva em: /workspace/pi-defense-exp/results/statistical_analysis/full/seed_stability.csv


,scenario_id,scenario_label,metric,direction,n,mean,std,range,coefficient_of_variation
0,c0_base,C0 — Base model,adjusted_win_rate_clean_vs_c0,higher,0,NaN,NaN,NaN,NaN
1,c0_base,C0 — Base model,clean_accuracy,higher,1,0.772921,NaN,0.0,NaN
2,c0_base,C0 — Base model,invalid_output_rate_all_attacks,lower,1,0.000067,NaN,0.0,NaN
3,c0_base,C0 — Base model,mr_seen,lower,1,0.810341,NaN,0.0,NaN
4,c0_base,C0 — Base model,mr_targeted_seen,lower,1,0.805011,NaN,0.0,NaN
5,c0_base,C0 — Base model,mr_targeted_unseen,lower,1,0.587420,NaN,0.0,NaN
6,c0_base,C0 — Base model,mr_unseen,lower,1,0.630419,NaN,0.0,NaN
7,c0_base,C0 — Base model,pna_i_seen,neutral,1,0.926119,NaN,0.0,NaN
8,c0_base,C0 — Base model,pna_i_unseen,neutral,1,0.897299,NaN,0.0,NaN
9,c0_base,C0 — Base model,robust_accuracy_all_attacks,higher,1,0.212886,NaN,0.0,NaN


## 15. Tabelas finais recomendadas para discussão

Esta seção monta tabelas menores, mais adequadas para leitura humana e discussão no texto final.

As tabelas aqui não substituem os arquivos completos. Elas apenas selecionam um subconjunto de métricas para facilitar interpretação.


In [15]:
DISCUSSION_METRICS = [
    "clean_accuracy",
    "utility_drop",
    "robust_accuracy_all_attacks",
    "targeted_asr_all_attacks",
    "untargeted_asr_all_attacks",
    "invalid_output_rate_all_attacks",
    "mr_targeted_seen",
    "mr_targeted_unseen",
    "adjusted_win_rate_clean_vs_c0",
]

available_discussion_metrics = [
    metric for metric in DISCUSSION_METRICS
    if metric in available_report_metrics
]

discussion_rows = []
for scenario_id in SCENARIO_ORDER:
    scenario_df = scenario_metric_summary_df[scenario_metric_summary_df["scenario_id"] == scenario_id]
    if scenario_df.empty:
        continue

    row = {
        "scenario_id": scenario_id,
        "scenario_label": SCENARIO_LABELS.get(scenario_id, scenario_id),
    }

    for metric in available_discussion_metrics:
        metric_match = scenario_df[scenario_df["metric"] == metric]
        if metric_match.empty:
            continue
        metric_row = metric_match.iloc[0]
        mean = metric_row["mean"]
        std = metric_row["std"]
        n = metric_row["n"]
        if pd.isna(mean):
            rendered = ""
        elif pd.isna(std):
            rendered = f"{mean:.4f}"
        else:
            rendered = f"{mean:.4f} ± {std:.4f}"
        row[metric] = rendered
        row[f"{metric}_n"] = int(n)

    discussion_rows.append(row)

discussion_table_df = pd.DataFrame(discussion_rows)
discussion_table_path = ANALYSIS_RESULTS_DIR / "discussion_table.csv"
discussion_table_md_path = ANALYSIS_RESULTS_DIR / "discussion_table.md"
discussion_table_df.to_csv(discussion_table_path, index=False)

with open(discussion_table_md_path, "w", encoding="utf-8") as f:
    f.write(dataframe_to_markdown_table(discussion_table_df))

print("Tabela de discussão salva em:", discussion_table_path)
display(discussion_table_df)


Tabela de discussão salva em: /workspace/pi-defense-exp/results/statistical_analysis/full/discussion_table.csv


,scenario_id,scenario_label,clean_accuracy,clean_accuracy_n,utility_drop,utility_drop_n,robust_accuracy_all_attacks,robust_accuracy_all_attacks_n,targeted_asr_all_attacks,targeted_asr_all_attacks_n,untargeted_asr_all_attacks,untargeted_asr_all_attacks_n,invalid_output_rate_all_attacks,invalid_output_rate_all_attacks_n,mr_targeted_seen,mr_targeted_seen_n,mr_targeted_unseen,mr_targeted_unseen_n,adjusted_win_rate_clean_vs_c0,adjusted_win_rate_clean_vs_c0_n
0,c0_base,C0 — Base model,0.7729,1,0.0000,1,0.2129,1,0.7818,1,0.7871,1,0.0001,1,0.8050,1,0.5874,1,,0
1,c1_struq_format_only,C1 — StruQ format-only,0.7735,1,-0.0005,1,0.2098,1,0.7852,1,0.7902,1,0.0004,1,0.5855,1,0.5737,1,0.4633,1
2,c2_struq_sft,C2 — StruQ-like SFT,0.8570 ± 0.0020,3,-0.0840 ± 0.0020,3,0.9837 ± 0.0014,3,0.0043 ± 0.0014,3,0.0163 ± 0.0014,3,0.0000 ± 0.0001,3,0.0000 ± 0.0000,3,0.0001 ± 0.0001,3,0.4449 ± 0.0083,3
3,c3_secalign_dpo,C3 — SecAlign-like DPO,0.7509 ± 0.0254,3,0.0220 ± 0.0254,3,0.9131 ± 0.0823,3,0.0679 ± 0.0855,3,0.0869 ± 0.0823,3,0.0050 ± 0.0054,3,0.0433 ± 0.0617,3,0.0522 ± 0.0578,3,0.4272 ± 0.0098,3
4,c4_ih_sft,C4 — Instruction-Hierarchy-like SFT,0.8564 ± 0.0055,3,-0.0835 ± 0.0055,3,0.9860 ± 0.0001,3,0.0005 ± 0.0003,3,0.0140 ± 0.0001,3,0.0000 ± 0.0000,3,0.0000 ± 0.0000,3,0.0003 ± 0.0005,3,0.4428 ± 0.0019,3


## 16. Conferência dos arquivos gerados

Antes de gerar o manifesto, verificamos se os principais artefatos deste notebook foram criados corretamente.


In [16]:
generated_files = {
    "unified_run_metrics_csv": unified_run_metrics_path,
    "scenario_metric_summary_csv": scenario_metric_summary_path,
    "scenario_compact_summary_csv": scenario_compact_summary_path,
    "deltas_vs_reference_csv": deltas_vs_reference_path,
    "metric_rankings_csv": ranking_path,
    "tradeoff_components_csv": tradeoff_components_path,
    "tradeoff_score_csv": tradeoff_score_path,
    "seed_stability_csv": stability_path,
    "discussion_table_csv": discussion_table_path,
    "discussion_table_md": discussion_table_md_path,
    "metric_catalog_csv": metric_catalog_path,
}

generated_file_rows = []
for name, path in generated_files.items():
    path = Path(path)
    if path.exists() and path.suffix == ".csv":
        rows = max(0, count_jsonl_lines(path) - 1)
    elif path.exists() and path.suffix == ".jsonl":
        rows = count_jsonl_lines(path)
    else:
        rows = None
    generated_file_rows.append({
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "rows": rows,
    })

generated_files_df = pd.DataFrame(generated_file_rows)
display(generated_files_df)

missing_generated = generated_files_df[~generated_files_df["exists"]]
if len(missing_generated) > 0:
    raise RuntimeError("Alguns arquivos esperados do notebook 08 não foram gerados.")

log_event("statistical_analysis_files_generated", {
    "num_files": int(len(generated_files_df)),
    "analysis_results_dir": str(ANALYSIS_RESULTS_DIR),
})


,name,path,exists,rows
0,unified_run_metrics_csv,/workspace/pi-defense-exp/results/statistical_...,True,11.0
1,scenario_metric_summary_csv,/workspace/pi-defense-exp/results/statistical_...,True,85.0
2,scenario_compact_summary_csv,/workspace/pi-defense-exp/results/statistical_...,True,5.0
3,deltas_vs_reference_csv,/workspace/pi-defense-exp/results/statistical_...,True,136.0
4,metric_rankings_csv,/workspace/pi-defense-exp/results/statistical_...,True,74.0
5,tradeoff_components_csv,/workspace/pi-defense-exp/results/statistical_...,True,35.0
6,tradeoff_score_csv,/workspace/pi-defense-exp/results/statistical_...,True,5.0
7,seed_stability_csv,/workspace/pi-defense-exp/results/statistical_...,True,85.0
8,discussion_table_csv,/workspace/pi-defense-exp/results/statistical_...,True,5.0
9,discussion_table_md,/workspace/pi-defense-exp/results/statistical_...,True,NaN


## 17. Gerar resumo e manifesto do notebook 08

O manifesto registra quais arquivos foram consumidos, quais análises foram feitas e quais artefatos foram produzidos.

Esse registro é importante para reprodução e para o notebook de exportação final, que poderá ler os arquivos consolidados sem precisar recalcular as análises.


In [17]:
summary = {
    "notebook": "08_statistical_analysis",
    "created_at_utc": utc_now(),
    "project_root": str(PROJECT_ROOT),
    "run_mode": ANALYSIS_RUN_MODE,
    "inputs": {
        "metrics_06_dir": str(METRICS_06_DIR),
        "pairwise_07_dir": str(PAIRWISE_07_DIR),
        "metrics_06_manifest": str(METRICS_MANIFEST_PATH),
        "pairwise_07_manifest": str(PAIRWISE_MANIFEST_PATH) if PAIRWISE_MANIFEST_PATH.exists() else None,
        "direct_run_metrics": str(DIRECT_RUN_METRICS_PATH),
        "pairwise_compact_metrics": str(PAIRWISE_COMPACT_PATH) if PAIRWISE_COMPACT_PATH.exists() else None,
    },
    "outputs": {name: str(path) for name, path in generated_files.items()},
    "logs": {
        "events_log_path": str(EVENTS_LOG_PATH),
        "summary_log_path": str(SUMMARY_LOG_PATH),
        "log_dir": str(LOG_DIR),
    },
    "counts": {
        "unified_run_rows": int(len(unified_run_metrics_df)),
        "scenario_metric_summary_rows": int(len(scenario_metric_summary_df)),
        "ranking_rows": int(len(ranking_df)),
        "delta_rows": int(len(deltas_vs_reference_df)),
        "tradeoff_score_rows": int(len(tradeoff_score_df)),
        "discussion_table_rows": int(len(discussion_table_df)),
    },
    "metrics_available": available_report_metrics,
    "metrics_used_for_discussion_table": available_discussion_metrics,
    "tradeoff_components": available_tradeoff_components,
    "notes": [
        "This notebook performs descriptive statistical consolidation only.",
        "Confidence intervals are based on the observed seeds and should be interpreted cautiously because the number of seeds is small.",
        "The exploratory trade-off score is not an official benchmark metric and should not replace individual metrics.",
        "PNA-I is treated as diagnostic/neutral in ranking because its interpretation depends on the experimental context.",
    ],
}

write_json(SUMMARY_LOG_PATH, summary)
print("Resumo salvo em:", SUMMARY_LOG_PATH)


Resumo salvo em: /workspace/pi-defense-exp/logs/statistical_analysis/08_statistical_analysis_summary_full.json


In [18]:
manifest_json_path = MANIFEST_DIR / "08_statistical_analysis_manifest.json"
manifest_md_path = MANIFEST_DIR / "08_statistical_analysis_manifest.md"

manifest = {
    **summary,
    "manifest_json_path": str(manifest_json_path),
    "manifest_md_path": str(manifest_md_path),
}

write_json(manifest_json_path, manifest)

output_table_md = dataframe_to_markdown_table(generated_files_df)
discussion_table_md = dataframe_to_markdown_table(discussion_table_df)
tradeoff_table_md = dataframe_to_markdown_table(tradeoff_score_df)

manifest_md = f'''# Manifesto — Notebook 08 Statistical Analysis

## Identificação

- Notebook: `08_statistical_analysis`
- Gerado em UTC: `{summary['created_at_utc']}`
- Run mode: `{ANALYSIS_RUN_MODE}`
- Diretório raiz do projeto: `{PROJECT_ROOT}`

## Entradas

- Métricas do notebook 06: `{METRICS_06_DIR}`
- Métricas do notebook 07: `{PAIRWISE_07_DIR}`
- Manifesto do notebook 06: `{METRICS_MANIFEST_PATH}`
- Manifesto do notebook 07: `{PAIRWISE_MANIFEST_PATH if PAIRWISE_MANIFEST_PATH.exists() else 'não encontrado'}`

## Métricas consolidadas

{', '.join(f'`{metric}`' for metric in available_report_metrics)}

## Arquivos gerados

{output_table_md}

## Tabela de discussão

{discussion_table_md}

## Score exploratório de trade-off

{tradeoff_table_md}

## Observações metodológicas

- Este notebook realiza consolidação estatística descritiva.
- Os intervalos de confiança são calculados a partir das seeds disponíveis e devem ser interpretados com cautela.
- O score de trade-off é exploratório e não substitui as métricas originais.
- Métricas como `pna_i` são tratadas como diagnósticas, não como ranking automático de melhor/pior defesa.
'''

with open(manifest_md_path, "w", encoding="utf-8") as f:
    f.write(manifest_md)

log_event("statistical_analysis_completed", {
    "manifest_json_path": str(manifest_json_path),
    "manifest_md_path": str(manifest_md_path),
})

print("Manifesto JSON salvo em:", manifest_json_path)
print("Manifesto Markdown salvo em:", manifest_md_path)


Manifesto JSON salvo em: /workspace/pi-defense-exp/manifests/statistical_analysis/08_statistical_analysis_manifest.json
Manifesto Markdown salvo em: /workspace/pi-defense-exp/manifests/statistical_analysis/08_statistical_analysis_manifest.md


## 18. Próximos passos

Este notebook consolidou os resultados quantitativos gerados pelos notebooks anteriores. A partir das métricas do `06_compute_metrics.ipynb` e, quando disponíveis, das métricas complementares do `07_pairwise_and_injected_metrics.ipynb`, foram produzidas tabelas agregadas por cenário, médias entre seeds, desvios padrão, intervalos de confiança aproximados, deltas em relação aos cenários de referência e rankings exploratórios.

Essa consolidação permite comparar os cenários experimentais de forma mais organizada:

```text id="octedl"
C0 — modelo base sem defesa
C1 — StruQ format-only
C2 — StruQ-like SFT
C3 — SecAlign-like DPO
C4 — Instruction-Hierarchy-like SFT
```

Os resultados deste notebook devem ser usados como base para a discussão quantitativa do experimento. Em especial, as tabelas consolidadas ajudam a identificar quais cenários preservam melhor a utilidade em exemplos limpos, quais reduzem melhor o sucesso dos ataques e quais apresentam melhor equilíbrio entre utilidade e robustez.

O próximo passo é complementar essa análise estatística com uma análise qualitativa de erros. Essa etapa será realizada no notebook:

```text id="brhj60"
09_error_analysis.ipynb
```

O objetivo do notebook 09 será inspecionar padrões de falha que não aparecem claramente nas métricas agregadas. Por exemplo:

```text id="g4rud9"
- exemplos em que uma defesa acerta e o modelo base erra;
- exemplos em que o modelo base resiste, mas uma defesa falha;
- ataques que produzem mais saídas inválidas;
- tasks mais sensíveis a ataques seen e unseen/adaptive;
- casos em que a defesa prejudica a utilidade limpa;
- falhas compartilhadas por todos os cenários;
- diferenças qualitativas entre C2, C3 e C4.
```

Essa análise qualitativa é importante porque as métricas agregadas indicam o desempenho geral, mas não explicam sozinhas por que certos cenários funcionam melhor ou pior. A inspeção de erros permite interpretar os resultados, identificar limitações das defesas e selecionar exemplos representativos para a discussão final do trabalho.

Depois da análise qualitativa, os resultados finais poderão ser exportados para tabelas, figuras e arquivos consolidados em um notebook posterior de exportação. Esse notebook de exportação deverá organizar os principais artefatos produzidos para facilitar a redação do relatório final.
